# Feature-level fusion M1 $+$ M2

Freeze 2 encoder, trích embedding CLS, nối feature tương tác của cả 2, train head MLP mới. Ngưỡng chốt trên val; test chỉ đánh giá cuối kèm McNemar so với M2 Best (0,9707).


In [ ]:
import base64, io, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
WORK = Path('/kaggle/working')
CLEAN = WORK / 'clean_fusion'
CLEAN.mkdir(parents=True, exist_ok=True)

def sh(cmd):
    print(f'\n$ {cmd}', flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with (WORK / 'live_progress.log').open('a', encoding='utf-8') as lf:
        for line in iter(p.stdout.readline, ''):
            print(line, end='', flush=True)
            lf.write(line); lf.flush()
    p.stdout.close()
    rc = p.wait()
    if rc != 0: raise RuntimeError(f'Fail rc={rc}: {cmd}')


In [ ]:
payload = '...'
repo_root = WORK / 'project'
repo_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(payload)), mode='r:gz') as tar:
    tar.extractall(repo_root)
os.chdir(repo_root)
print('Repo:', os.getcwd(), flush=True)


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available(): raise SystemError('Chua bat GPU!')
sh('pip install -q -U transformers datasets scikit-learn scipy pyarrow')


In [ ]:
data_raw = repo_root / 'data/raw/bigclonebench'
data_raw.mkdir(parents=True, exist_ok=True)
ckpts = {
    'm1_best.pt': 'results/exp_m1_lr/best_checkpoint.pt',
    'm2_best.pt': 'results/exp_m2_dr/best_checkpoint.pt',
}
for name, dest in ckpts.items():
    cands = list(Path('/kaggle/input').rglob(name))
    p = repo_root / dest
    p.parent.mkdir(parents=True, exist_ok=True)
    if cands and not p.exists(): sh(f'cp "{cands[0]}" "{p}"')
    print(dest, p.exists(), flush=True)
    assert p.exists(), f'Thieu {dest}'
# Copy cache embedding để script bỏ qua bước trích (tiết kiệm ~24 phút)
emb_dir = repo_root / 'results/exp_fusion_m1m2'
emb_dir.mkdir(parents=True, exist_ok=True)
for f in Path('/kaggle/input').rglob('*.npy'):
    dest = emb_dir / f.name
    if not dest.exists(): shutil.copy2(f, dest)
print('cached npy:', len(list(emb_dir.glob('*.npy'))), flush=True)

for name in ('bcb_train.parquet','bcb_validation.parquet','bcb_test.parquet'):
    cands = list(Path('/kaggle/input').rglob(name))
    dest = data_raw / name
    if cands and not dest.exists(): sh(f'cp "{cands[0]}" "{dest}"')
    print(name, dest.exists(), flush=True)
    assert dest.exists(), f'Thieu {name}'


In [ ]:
print('=' * 70)
print('>>> BAT DAU FUSION M1 + M2 <<<')
print('=' * 70)
t0 = time.perf_counter()
sh('python scripts/run_fusion_m1m2.py --batch-size 32 --max-length 512 --epochs 50')
print(f'>>> XONG TRONG {(time.perf_counter()-t0)/60:.1f} PHUT', flush=True)
print((repo_root / 'results/exp_fusion_m1m2/report.json').read_text(encoding='utf-8'), flush=True)


In [ ]:
out_dir = repo_root / 'results/exp_fusion_m1m2'
for p in out_dir.rglob('*'):
    if p.is_file() and p.suffix in ('.json', '.npy', '.pt'):
        name = 'fusion_' + p.name
        shutil.copy2(p, CLEAN / name)
        shutil.copy2(p, WORK / name)
        print('  [SAVED]', name, flush=True)
log = WORK / 'live_progress.log'
if log.exists():
    shutil.copy2(log, CLEAN / 'fusion_live_progress.log')
    shutil.copy2(log, WORK / 'fusion_live_progress.log')
with tarfile.open(WORK / 'fusion_bundle.tar.gz', 'w:gz') as tar:
    for f in CLEAN.glob('*'): tar.add(f, arcname=f.name)
os.chdir(WORK)
for junk in ('project', 'data', 'clean_fusion'):
    shutil.rmtree(WORK / junk, ignore_errors=True)
print('>>> XONG 100%!', flush=True)